# Notebook 1 [KAGGLE] — YOLO11n Drone Fine-Tuning [Phases 1 & 2]
**Model:** YOLO11n | **Runtime:** GPU T4 x2 (preferred) or P100  
**Phases covered:** Training → Validation → Outputs saved to `/kaggle/working/`

> 📄 [YOLO11 model overview](https://docs.ultralytics.com/models/yolo11/)  
> 📄 [Ultralytics train() arguments](https://docs.ultralytics.com/modes/train/#train-settings)  
> 📄 [Dataset — seraphim-drone-detection](https://huggingface.co/datasets/lgrzybowski/seraphim-drone-detection-dataset)

---
## Key Kaggle differences from Colab

| | Colab | Kaggle |
|---|---|---|
| Output files | `files.download()` to browser | Auto-appear in **Output tab** (right panel) |
| Checkpoint resume | Upload `.pt` via file prompt | Add `.pt` as **Kaggle Dataset input** |
| HF token | Colab Secrets | Kaggle Secrets (Add-ons → Secrets) |
| Working directory | `/content/` | `/kaggle/working/` |
| GPU quota | ~4 hrs/day free | **30 hrs/week** free |
| Session length | ~12 hrs | **12 hrs** |

---
## Daily Workflow
```
Day 1 (fresh):  RESUME_MODE=False, EPOCH_OFFSET=0  → Run All
                → download outputs from Output tab (right panel)

Day 2+:         Upload checkpoint as Kaggle Dataset (see A1.3 instructions)
                RESUME_MODE=True, EPOCH_OFFSET=5   → Run All

Final session:  Run All → A5 validates → hand best.pt to Notebook 2

EPOCH_OFFSET schedule (5 epochs/day):
  Day 1: 0 | Day 2: 5 | Day 3: 10 | Day 4: 15 | Day 5: 20 | Day 6: 25
```

---
## Pipeline Position
```
[Notebook 1 — Kaggle]                  [Notebook 2]
yolo11n.pt → fine-tune → best.pt  →   upload best.pt → MCT → Compile → Package → packerOut.zip
             validate ↑
```

---
## A0 — Configuration (only section you edit each session)

In [ ]:
# A0 — Config
# ─────────────────────────────────────────────────────────────────
# RESUME_MODE:
#   False → Day 1: fresh start from yolo11n.pt (COCO pretrained)
#   True  → Day 2+: loads checkpoint from Kaggle Dataset input (see A1.3)
#
# CHECKPOINT_DATASET_NAME:
#   The name you gave your checkpoint dataset on Kaggle.
#   Kaggle mounts it at: /kaggle/input/{CHECKPOINT_DATASET_NAME}/
#   Example: if you named it 'drone-checkpoint', set below accordingly.
#
# EPOCH_OFFSET:
#   Total epochs completed BEFORE this session.
#   Day 1: 0 | Day 2: 5 | Day 3: 10 | Day 4: 15 | Day 5: 20 | Day 6: 25
# ─────────────────────────────────────────────────────────────────

RESUME_MODE              = False             # ← change to True from Day 2 onwards
EPOCHS_THIS_SESSION      = 200                 # ← epochs to run today
EPOCH_OFFSET             = 0                 # ← total epochs completed before today
CHECKPOINT_DATASET_NAME  = 'drone-checkpoint'# ← your Kaggle dataset name (resume only)

# Fixed config — do not change
import os
MODEL_NAME  = 'yolo11_drone'
PROJECT_DIR = '/kaggle/working/drone_finetune'
DATASET_DIR = '/kaggle/working/drone_dataset'
YAML_PATH   = '/kaggle/working/data.yaml'
WEIGHTS_DIR = f'{PROJECT_DIR}/{MODEL_NAME}/weights'
LAST_PT     = f'{WEIGHTS_DIR}/last.pt'
BEST_PT     = f'{WEIGHTS_DIR}/best.pt'
RESULTS_PNG = f'{PROJECT_DIR}/{MODEL_NAME}/results.png'

# Checkpoint input path (Kaggle mounts datasets under /kaggle/input/)
CHECKPOINT_INPUT_DIR = f'/kaggle/input/datasets/{kaggle_username}/{CHECKPOINT_DATASET_NAME}'
# /kaggle/input/datasets/kaggle_username/{CHECKPOINT_DATASET_NAME}/best.pt
os.makedirs(PROJECT_DIR, exist_ok=True)

print(f'✓ Config loaded')
print(f'  Mode              : {"RESUME" if RESUME_MODE else "FRESH START"}')
print(f'  Epochs today      : {EPOCHS_THIS_SESSION}')
print(f'  Epoch offset      : {EPOCH_OFFSET}')
print(f'  Working directory : /kaggle/working/')
if RESUME_MODE:
    print(f'  Checkpoint input  : {CHECKPOINT_INPUT_DIR}')

---
## A1 — Install + GPU check + Load checkpoint

### A1.3 — How to add checkpoint for resume (Day 2+)
Kaggle does not have a file upload prompt. Instead:
1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
2. Upload your `checkpoint_epoch_XX.pt` file
3. Name it exactly: **`drone-checkpoint`** (or whatever you set in `CHECKPOINT_DATASET_NAME` above)
4. Set visibility to **Private**
5. Back in this notebook → **Add Input** (top right) → search `drone-checkpoint` → Add
6. Kaggle mounts it at `/kaggle/input/drone-checkpoint/`
7. Set `RESUME_MODE=True` in A0 and run

In [ ]:
# A1.1 — Install packages
# Kaggle pre-installs torch/CUDA. Only ultralytics + huggingface_hub needed.
!pip install ultralytics huggingface_hub --quiet
print('✓ Packages installed')

In [ ]:
# A1.2 — Verify GPU
import torch
if torch.cuda.is_available():
    print(f'✓ GPU   : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'  CUDA  : {torch.version.cuda}')
    print(f'  Python: {__import__("sys").version.split()[0]}')
else:
    print('✗ No GPU — STOP. Settings (right panel) → Accelerator → GPU T4 x2')

In [ ]:
# A1.3 — Load checkpoint (RESUME MODE only)
# Day 1: skipped automatically.
# Day 2+: checkpoint is read from Kaggle Dataset input (see instructions above).
#
# WHY resume=False + explicit weight load (not resume=True):
#   resume=True reads the internal epoch counter from the .pt file
#   → wrong epoch numbering across sessions.
#   Loading checkpoint as weights with resume=False resets the session
#   counter correctly.

import shutil, glob, os

if RESUME_MODE:
    # Find the checkpoint file inside the Kaggle dataset input folder
    checkpoint_files = glob.glob(f'{CHECKPOINT_INPUT_DIR}/*.pt')

    print(checkpoint_files)
    if not checkpoint_files:
        raise FileNotFoundError(
            f'No .pt file found in {CHECKPOINT_INPUT_DIR}\n'
            f'  → Add your checkpoint as a Kaggle Dataset input (see A1.3 instructions above)'
        )

    # Use the most recently modified checkpoint if multiple exist
    latest = max(checkpoint_files, key=os.path.getmtime)
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    shutil.copy(latest, LAST_PT)
    print(f'✓ Checkpoint loaded: {latest}')
    print(f'  Copied to        : {LAST_PT}')
else:
    print('Fresh start — skipping checkpoint load.')

---
## A2 — Dataset

In [ ]:
# A2.1 — HuggingFace token
# Kaggle → Add-ons → Secrets → Add Secret → Name: HF_TOKEN
# Then enable it for this notebook via the Secrets panel.
# Dataset is public — works without token but token avoids rate-limit warnings.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print('✓ HF_TOKEN loaded from Kaggle Secrets')
except Exception:
    import os
    hf_token = os.environ.get('HF_TOKEN', None)
    print('ℹ HF_TOKEN not in Kaggle Secrets — using env or unauthenticated')

In [ ]:
# A2.2 — Download dataset from HuggingFace (~5 min)
from huggingface_hub import snapshot_download
print('Downloading seraphim-drone-detection-dataset...')
snapshot_download(
    repo_id  = 'lgrzybowski/seraphim-drone-detection-dataset',
    repo_type= 'dataset',
    local_dir= DATASET_DIR,
    token    = hf_token
)
print('\n✓ Download complete')

In [ ]:
# A2.3 — Extract zip batches
import zipfile, glob, os
zip_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.zip'), recursive=True)
print(f'Found {len(zip_files)} zip files. Extracting...')
for i, zp in enumerate(sorted(zip_files)):
    with zipfile.ZipFile(zp, 'r') as zf:
        zf.extractall(os.path.dirname(zp))
    os.remove(zp)
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(zip_files)}...')
print('\n✓ Extraction complete')

In [ ]:
# A2.4 — Verify image/label counts match
for split in ['train', 'test']:
    imgs   = glob.glob(os.path.join(DATASET_DIR, split, 'images', '*.jpg'))
    labels = glob.glob(os.path.join(DATASET_DIR, split, 'labels', '*.txt'))
    ok = '✓' if len(imgs) == len(labels) and len(imgs) > 0 else '✗ MISMATCH'
    print(f'{ok}  {split:5s}  images={len(imgs):6,}  labels={len(labels):6,}')

---
## A3 — Write data.yaml

In [ ]:
# A3 — Write data.yaml
import yaml
data_config = {
    'path' : DATASET_DIR,
    'train': 'train/images',
    'val'  : 'test/images',
    'nc'   : 1,
    'names': ['drone']
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)
print('✓ data.yaml written')
with open(YAML_PATH) as f: print(f.read())

---
## A4 — Fine-Tune

**Kaggle output note:**  
No `files.download()` needed. All files saved to `/kaggle/working/` appear automatically  
in the **Output tab** (right panel) at session end. Download from there.

**Checkpoint copies** are saved to `/kaggle/working/` after every epoch.  
Download the latest one → upload as new version of your `drone-checkpoint` Kaggle Dataset → resume tomorrow.

> 📄 [train() arguments reference](https://docs.ultralytics.com/modes/train/#train-settings)

In [ ]:
# A4 — Training
import gc, os, shutil, torch
from datetime import datetime
from ultralytics import YOLO

def on_epoch_end(trainer):
    """
    After every epoch:
      1. GC + CUDA flush — prevents RAM leak across long sessions
      2. Print timestamp + offset-corrected epoch number
      3. Copy numbered checkpoint to /kaggle/working/ — appears in Output tab
    """
    gc.collect()
    torch.cuda.empty_cache()

    current_epoch = trainer.epoch + 1 + EPOCH_OFFSET
    timestamp     = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'  [GC]   Epoch {current_epoch:02d} complete — RAM flushed — {timestamp}')

    if not os.path.exists(LAST_PT):
        print(f'  [CKPT] ⚠ last.pt not found — skipping')
        return

    # Save checkpoint to /kaggle/working/ — visible in Output tab
    dest = f'/kaggle/working/checkpoint_epoch_{current_epoch:02d}.pt'
    shutil.copy(LAST_PT, dest)
    print(f'  [CKPT] ✓ checkpoint_epoch_{current_epoch:02d}.pt → Output tab')
    print(f'  [CKPT]   Download from Output tab → upload as drone-checkpoint dataset to resume')


# Load model
# resume=False + explicit weight load = correct session-based resume
# resume=True reads internal epoch counter from .pt → wrong epoch numbering
if RESUME_MODE:
    print(f'WEIGHT TRANSFER — loading: {LAST_PT}')
    model = YOLO(LAST_PT)
else:
    print('FRESH START — loading yolo11n.pt (COCO pretrained, ~5MB)')
    model = YOLO('yolo11n.pt')

model.add_callback('on_fit_epoch_end', on_epoch_end)

results = model.train(
    data     = YAML_PATH,
    epochs   = EPOCHS_THIS_SESSION,
    imgsz    = 640,
    batch    = 64,   # Kaggle P100 has 16GB VRAM — can try 64 if P100 assigned
    device   = 0,
    cache    = 'disk',
    workers  = 4,    # Kaggle has more CPU cores than Colab free tier
    project  = PROJECT_DIR,
    name     = MODEL_NAME,
    exist_ok = True,
    resume   = False,
    patience = 15,
    save     = True,
    plots    = True,
    verbose  = True
)

# ── Copy safety files to /kaggle/working/ root for easy access in Output tab ──
import shutil
shutil.copy(BEST_PT,     '/kaggle/working/best.pt')
shutil.copy(RESULTS_PNG, '/kaggle/working/results.png')
print('\n✓ best.pt and results.png → Output tab')
print(f'→ Next session: RESUME_MODE=True, EPOCH_OFFSET={EPOCH_OFFSET + EPOCHS_THIS_SESSION}')
print('→ Upload latest checkpoint_epoch_XX.pt as new version of drone-checkpoint dataset')

---
## A5 — Validate (final session only)
Run after last training session. Confirm mAP before handing `best.pt` to Notebook 2.

| mAP@0.5 | Decision |
|---|---|
| ≥ 0.90 | Excellent — proceed to Notebook 2 |
| ≥ 0.70 | Acceptable — proceed to Notebook 2 |
| < 0.70 | Train more epochs |

> 📄 [val() reference](https://docs.ultralytics.com/modes/val/)

In [ ]:
# A5 — Validate best.pt
from ultralytics import YOLO
best_model = YOLO(BEST_PT)
metrics    = best_model.val(data=YAML_PATH, split='val', device=0)
map50      = metrics.box.map50
map5095    = metrics.box.map

print(f'\n✓ Validation complete')
print(f'  mAP@0.5      : {map50:.4f}')
print(f'  mAP@0.5:0.95 : {map5095:.4f}')

if map50 >= 0.90:
    print('  ✓ Excellent — download best.pt from Output tab → hand to Notebook 2')
elif map50 >= 0.70:
    print('  ✓ Acceptable — proceed to Notebook 2')
else:
    print('  ⚠ Below target — consider more training epochs')
    

A5.1 - export ONNX

In [ ]:
# A5.1 — Export to ONNX
from ultralytics import YOLO
best_model = YOLO(BEST_PT)
best_model.export(format='onnx', imgsz=640, simplify=True, opset=16)
print(f'✓ Export complete: {BEST_PT.replace(".pt", ".onnx")}')

---
## A6 — Zip all session checkpoints (optional)
Bundles all epoch checkpoints into one zip in `/kaggle/working/` → visible in Output tab.

In [ ]:
# A6 — Zip session checkpoints
import zipfile, os

zip_filename = '/kaggle/working/checkpoints.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for i in range(EPOCH_OFFSET + 1, EPOCH_OFFSET + 1 + EPOCHS_THIS_SESSION):
        f = f'/kaggle/working/checkpoint_epoch_{i:02d}.pt'
        if os.path.exists(f):
            zipf.write(f, os.path.basename(f))
        else:
            print(f'  Warning: {f} not found')

print(f'✓ checkpoints.zip saved → Output tab')

In [ ]:
# ls -lh /kaggle/working/

---
## Quick Reference
```
Every session:
  A0: set RESUME_MODE, EPOCHS_THIS_SESSION, EPOCH_OFFSET
  Run All → outputs appear in Output tab (right panel)
  Download checkpoint_epoch_XX.pt from Output tab
  Upload as new version of drone-checkpoint Kaggle Dataset

Resume setup (before Day 2 run):
  kaggle.com/datasets → drone-checkpoint → New Version → upload latest .pt
  This notebook → Add Input → drone-checkpoint → Add
  A0: RESUME_MODE=True, EPOCH_OFFSET=+5

Final session:
  Run All → A5 validates
  Download best.pt from Output tab → hand to Notebook 2

Output tab location:
  Right panel → Output → /kaggle/working/
  Files available for download after session or during run
```

---
## Key References
| Topic | Link |
|---|---|
| YOLO11 model | https://docs.ultralytics.com/models/yolo11/ |
| train() arguments | https://docs.ultralytics.com/modes/train/#train-settings |
| val() reference | https://docs.ultralytics.com/modes/val/ |
| Drone dataset | https://huggingface.co/datasets/lgrzybowski/seraphim-drone-detection-dataset |
| Kaggle Datasets docs | https://www.kaggle.com/docs/datasets |